In [12]:
import pandas as pd
import requests
import time
import os

In [13]:
# Google Maps API Setup - Uses environment variable (NEVER hardcode keys!)
import os
import requests
import json

GOOGLE_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY")

if not GOOGLE_API_KEY:
    print("⚠️ GOOGLE_MAPS_API_KEY not set in environment")
    print("   Run: export GOOGLE_MAPS_API_KEY='your_key_here'")
else:
    print(f"✅ Google Maps API key loaded (ends with: ...{GOOGLE_API_KEY[-4:]})")

def geocode_place(place_name, api_key=GOOGLE_API_KEY):
    """Convert place name to lat/lon using Google Geocoding API."""
    if not api_key:
        return None
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": place_name, "key": api_key}
    r = requests.get(url, params=params, timeout=10)
    data = r.json()
    if data["status"] == "OK" and data["results"]:
        loc = data["results"][0]["geometry"]["location"]
        return {
            "place_name": place_name,
            "formatted_address": data["results"][0]["formatted_address"],
            "lat": loc["lat"],
            "lng": loc["lng"],
            "place_id": data["results"][0]["place_id"]
        }
    print(f"❌ Geocoding failed: {data['status']}")
    return None

# Example usage (only runs if key is set):
# result = geocode_place("Vík í Mýrdal, Iceland")
# print(json.dumps(result, indent=2))


✅ Google Maps API key loaded (ends with: ...HP3Y)


In [ ]:
# set google maps api key here once when developing 
# os.environ["GOOGLE_MAPS_API_KEY"] = "" # replace here key and run cell

In [ ]:
# Google Maps Directions API - Get route with legs, distances, durations
import os
import requests
import json
import re
from urllib.parse import unquote

GOOGLE_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY")

def get_directions(origin, destination, waypoints=None, api_key=GOOGLE_API_KEY):
    """Get driving directions using Google Directions API.
    
    Args:
        origin: Start point ("lat,lng" or place name)
        destination: End point ("lat,lng" or place name)
        waypoints: List of intermediate points ["lat,lng", ...] or None
        api_key: Google Maps API key
    
    Returns:
        Dict with route info: legs, total_distance, total_duration, polyline
    """
    if not api_key:
        print("❌ No API key")
        return None
    
    url = "https://maps.googleapis.com/maps/api/directions/json"
    params = {
        "origin": origin,
        "destination": destination,
        "mode": "driving",
        "key": api_key
    }
    if waypoints:
        # Format: via:lat,lng|via:lat,lng
        wp_str = "|".join([f"via:{wp}" for wp in waypoints])
        params["waypoints"] = wp_str
    
    r = requests.get(url, params=params, timeout=15)
    data = r.json()
    
    if data["status"] != "OK" or not data["routes"]:
        print(f"❌ Directions failed: {data['status']}")
        if "error_message" in data:
            print(f"   {data['error_message']}")
        return None
    
    route = data["routes"][0]
    leg = route["legs"][0]
    
    # Extract all leg details
    legs = []
    total_distance_m = 0
    total_duration_s = 0
    
    for i, leg in enumerate(route["legs"]):
        legs.append({
            "leg_index": i,
            "start_address": leg["start_address"],
            "end_address": leg["end_address"],
            "start_location": leg["start_location"],  # {lat, lng}
            "end_location": leg["end_location"],
            "distance_m": leg["distance"]["value"],
            "distance_text": leg["distance"]["text"],
            "duration_s": leg["duration"]["value"],
            "duration_text": leg["duration"]["text"],
            "steps": len(leg["steps"])
        })
        total_distance_m += leg["distance"]["value"]
        total_duration_s += leg["duration"]["value"]
    
    return {
        "total_distance_km": round(total_distance_m / 1000, 2),
        "total_duration_hours": round(total_duration_s / 3600, 2),
        "total_distance_text": f"{total_distance_m/1000:.1f} km",
        "total_duration_text": f"{total_duration_s/3600:.1f} hours",
        "legs": legs,
        "overview_polyline": route["overview_polyline"]["points"]
    }

def parse_google_maps_url(gmaps_url, api_key=GOOGLE_API_KEY):
    """Parse a long Google Maps URL (maps/dir/...) to extract origin, destination, waypoints.
    
    Example input:
    https://www.google.com/maps/dir/Egilsstaðir,+700,+Iceland/Mývatn,+660,+Iceland/@65.4462439,-17.0877355,7.6z/data=...
    
    Returns dict with: origin, destination, waypoints (list), raw_places (list)
    """
    if not api_key:
        print("❌ No API key")
        return None
    
    # Extract the path between /maps/dir/ and /@ or /data=
    if "/maps/dir/" not in gmaps_url:
        print(f"❌ Not a valid Google Maps directions URL")
        return None
    
    # Get everything after /maps/dir/
    path_part = gmaps_url.split("/maps/dir/")[1]
    
    # Remove viewport (@...) and data (=...) parts
    path_part = path_part.split("/@")[0]
    path_part = path_part.split("/data=")[0]
    
    # Split by / to get individual places
    raw_places = [p for p in path_part.split("/") if p]
    
    # URL decode each place
    places = [unquote(p.replace("+", " ")) for p in raw_places]
    
    if len(places) < 2:
        print(f"❌ Need at least origin and destination, got: {places}")
        return None
    
    origin = places[0]
    destination = places[-1]
    waypoints = places[1:-1] if len(places) > 2 else None
    
    return {
        "origin": origin,
        "destination": destination,
        "waypoints": waypoints,
        "raw_places": places
    }



In [21]:
# Example usage with your long URL (only runs if key is set):
long_url = "https://www.google.com/maps/dir/Egilsstaðir,+700,+Iceland/Mývatn,+660,+Iceland/@65.4462439,-17.0877355,7.6z/data=!4m14!4m13!1m5!1m1!1s0x48cc04800da89ed3:0x95bf416b2c7c7f54!2m2!1d-14.3994394!2d65.2609232!1m5!1m1!1s0x48cd9c44953c07dd:0xcde4cb0dbf732a88!2m2!1d-16.9961055!2d65.60386!3e0?entry=ttu&g_ep=EgoyMDI2MDcyOS4wIKXMDSoASAFQAw%3D%3D"
route_info = parse_google_maps_url(long_url)
if route_info:
    print(json.dumps(route_info, indent=2))
    directions = get_directions(
        origin=route_info["origin"],
        destination=route_info["destination"],
        waypoints=route_info["waypoints"]
    )
    print(json.dumps(directions, indent=2))

{
  "origin": "Egilssta\u00f0ir, 700, Iceland",
  "destination": "M\u00fdvatn, 660, Iceland",
  "waypoints": null,
  "raw_places": [
    "Egilssta\u00f0ir, 700, Iceland",
    "M\u00fdvatn, 660, Iceland"
  ]
}
{
  "total_distance_km": 174.17,
  "total_duration_hours": 2.14,
  "total_distance_text": "174.2 km",
  "total_duration_text": "2.1 hours",
  "legs": [
    {
      "leg_index": 0,
      "start_address": "700 Egilssta\u00f0ir, Iceland",
      "end_address": "M\u00fdvatn, 660, Iceland",
      "start_location": {
        "lat": 65.26092779999999,
        "lng": -14.3994208
      },
      "end_location": {
        "lat": 65.615821,
        "lng": -17.0244046
      },
      "distance_m": 174168,
      "distance_text": "174 km",
      "duration_s": 7714,
      "duration_text": "2 hours 9 mins",
      "steps": 5
    }
  ],
  "overview_polyline": "ygimKjk{vAN`[{@~RgCFmWgKuJh@u[|_@w]p]cMhZoZjr@ie@~d@s[|d@g[tv@sDrn@wJ|VgXbHsOdRge@pR}bBp}Aeg@tq@cs@l^mc@z_@w`@v`@gShkAqSrg@mTnIg[pBkd@bM{a@`CgN

In [20]:
# Placeholder station IDs - later populate with Google Maps integration
# These are real IMO station IDs along/near Ring Road
station_ids = [
    1350,  # Keflavíkurflugvöllur (Reykjavík area)
    1475  # Reykjavík
    
]

weather_api = "https://api.vedur.is/weather"
url = f"{weather_api}/observations/aws/hour/latest"

all_station_data = []

for station_id in station_ids:
    params = {
        "station_id": station_id,
        "parameters": "all",
        "format": "json",
        "day_from": "2026-08-03",
        "day_to": "2026-08-03",
    }
    
    try:
        result = requests.get(url, params=params, timeout=10)
        result.raise_for_status()
        data = result.json()
        
        if data:  # Only add if data returned
            # API already returns 'station' field with the station ID
            # Do NOT manually add station_id - it creates duplicates after rename
            all_station_data.extend(data)
            print(f"✅ Station {station_id}: {len(data)} records")
        else:
            print(f"⚠️ Station {station_id}: No data returned")
            
    except requests.exceptions.RequestException as e:
        print(f"❌ Station {station_id}: Request failed - {e}")
    except Exception as e:
        print(f"❌ Station {station_id}: Error - {e}")
    
    time.sleep(2)  

print(f"\nTotal records collected: {len(all_station_data)}")


# Convert to DataFrame for inspection
df_weather = pd.DataFrame(all_station_data)
df_weather.head()

✅ Station 1350: 1 records
✅ Station 1475: 1 records

Total records collected: 2


,station,name,time,year,month,day,hour,t,tx,tn,...,radsws,radswsx,radlwi,radlwix,radlws,radlwsx,raduv,raduvx,rsun,count_measurements
0,1350,Keflavíkurflugvöllur,2026-08-04T23:00:00,2026,8,4,23,9.7,9.7,9.5,...,None,None,None,None,None,None,None,None,None,6
1,1475,Reykjavík - Bústaðavegur,2026-08-04T23:00:00,2026,8,4,23,10.2,10.2,10.1,...,None,None,None,None,None,None,None,None,None,6


## Vedur API Data Dictionary

Column mappings from IMO (Icelandic Meteorological Office) AWS hourly observations API.
Source: https://api.vedur.is/weather/observations/aws/hour

In [9]:
# Vedur API column name -> full description mapping
VEDUR_COLUMN_MAP = {
    # Station metadata
    "station": "station_id",
    "name": "station_name",
    
    # Time dimensions
    "time": "observation_time_utc",
    "year": "year",
    "month": "month",
    "day": "day",
    "hour": "hour_utc",  # 1-24, where 24 = midnight
    
    # Temperature (Celsius)
    "t": "air_temp_c",           # Air temperature at observation time
    "tx": "air_temp_max_c",      # Maximum air temperature since last observation
    "tn": "air_temp_min_c",      # Minimum air temperature since last observation
    
    # Humidity & moisture
    "rh": "relative_humidity_pct",  # Relative humidity (%)
    "vp": "vapor_pressure_hpa",     # Vapor pressure (hPa)
    "td": "dew_point_c",            # Dew point temperature (C)
    
    # Wind
    "f": "wind_speed_avg_ms",       # Average wind speed (m/s) over 10-min period
    "fx": "wind_speed_max_ms",      # Maximum 10-min average wind speed (m/s)
    "fg": "wind_gust_max_ms",       # Maximum wind gust (m/s)
    "fgfx": "gust_factor",          # Ratio of max gust to max 10-min wind (fg/fx)
    "d": "wind_dir_deg",            # Wind direction (degrees, 0-360)
    "d_txt": "wind_dir_cardinal",   # Wind direction (cardinal: N, NE, E, etc.)
    "dsdev": "wind_dir_std_dev",    # Standard deviation of wind direction (degrees)
    
    # Pressure
    "ps": "station_pressure_hpa",   # Station level pressure (hPa)
    "p": "sea_level_pressure_hpa",  # Sea level pressure (hPa)
    
    # Precipitation
    "r": "precipitation_mm",        # Precipitation since last observation (mm)
    
    # Ground temperature
    "tg": "ground_temp_c",          # Ground temperature (C)
    "tgn": "ground_temp_min_c",     # Minimum ground temperature (C)
    
    # Road surface temperature (for road stations)
    "t0": "road_surface_temp_c",    # Road surface temperature (C)
    "t0x": "road_surface_temp_max_c",
    "t0n": "road_surface_temp_min_c",
    
    # Turf/grass temperature at various depths
    "tug5": "turf_temp_5cm_c",
    "tug10": "turf_temp_10cm_c",
    "tug15": "turf_temp_15cm_c",
    "tug20": "turf_temp_20cm_c",
    "tug50": "turf_temp_50cm_c",
    "tug100": "turf_temp_100cm_c",
    
    # Radiation
    "radgl": "global_radiation_wm2",       # Global radiation (W/m²)
    "radglx": "global_radiation_max_wm2",
    "radsc": "shortwave_radiation_wm2",    # Shortwave radiation (W/m²)
    "radscx": "shortwave_radiation_max_wm2",
    "radsws": "sunshine_duration_s",       # Sunshine duration (seconds)
    "radswsx": "sunshine_duration_max_s",
    "radlwi": "longwave_in_radiation_wm2", # Longwave incoming radiation (W/m²)
    "radlwix": "longwave_in_radiation_max_wm2",
    "radlws": "longwave_out_radiation_wm2", # Longwave outgoing radiation (W/m²)
    "radlwsx": "longwave_out_radiation_max_wm2",
    "raduv": "uv_radiation_wm2",           # UV radiation (W/m²)
    "raduvx": "uv_radiation_max_wm2",
    
    # Other
    "rsun": "sunshine_duration_min",       # Sunshine duration (minutes)
    "ts": "snow_depth_cm",                 # Snow depth (cm)
    "sal": "salinity_psu",                 # Salinity (PSU) - for marine stations
    "count_measurements": "measurement_count",  # Number of measurements in period
}

# Print nicely formatted
for short, full in VEDUR_COLUMN_MAP.items():
    print(f"  {short:>15} -> {full}")

In [ ]:
# Rename columns in df_weather using the data dictionary
df_weather_renamed = df_weather.rename(columns=VEDUR_COLUMN_MAP)

print("Original columns:")
print(df_weather.columns.tolist())
print("\nRenamed columns:")
print(df_weather_renamed.columns.tolist())

# Check for duplicates
dupes = df_weather_renamed.columns[df_weather_renamed.columns.duplicated()].tolist()
if dupes:
    print(f"\n⚠️ DUPLICATE COLUMNS: {dupes}")
else:
    print("\n✅ No duplicate columns")

# Show sample with new names
df_weather_renamed.head()

In [ ]:
# Save renamed data for DuckDB pipeline
os.makedirs("data", exist_ok=True)

df_weather_renamed.to_parquet("data/iceland_weather_renamed.parquet", index=False)
df_weather_renamed.to_csv("data/iceland_weather_renamed.csv", index=False)

print("Saved to data/iceland_weather_renamed.parquet and .csv")
print(f"File size: {os.path.getsize('data/iceland_weather_renamed.parquet') / 1024:.1f} KB")